### Scheda di acquisizione

**Steps**

1- initialize hardware (scheda, oscilloscopio)  
2- set CHANNELS to watch on oscilloscope  
3- set TRIGGER  
4- set function generator
3- do acquisition 
4- plot data
5- export data

In [ ]:
# Initialize the hardware
import tdwf
ad2 = tdwf.AD2(iconfig=1)
scope = tdwf.Scope(ad2.hdwf)
scope.fs = 1e6
scope.npt = 1000

# Channels
scope.ch1.rng = 5
scope.ch1.offs = 3.0
scope.ch2.rng = 50
scope.ch2.avg = True

# Trigger
scope.trig(True, level = 1, hist = 0.1, sour = tdwf.trigsrcCh2)

# Supply
ad2.vdd = +5   # (red wire ???)
ad2.vss = -3  # (white wire???)
ad2.power(True)

# Function generator
wgen = tdwf.WaveGen(ad2.hdwf)
wgen.w1.offs = 1
wgen.w1.func = tdwf.funcSine
# wgen.w1.config(offs=1, func=tdwf.funcDC)  one-liner
wgen.w1.start()
# sleep ???

# Do acquisition
scope.sample()
print(scope.ch1.vals)
ad2. close()
# Plot data
import matplotlib.pyplot as plt
plt.plot(scope.time.vals, scope.ch1.vals)
plt.xlabel("Tempo [s]")
plt.ylabel("Ch1 [V]")

# Export data
import numpy as npt
data = np.column_stack((
    scope.time.vals,
    scope.ch1.vals,
    scope.ch2.vals
))
header = f"Acquisition timestamp: {scope.time.t0}\nTime\tChannel 1\tChannel 2"
np.savetxt(
    "nomefile.txt",
    data,
    delimiter="\t",
    header=header,
    fmt="%.8e"    // 8 decimal digits
)


**Hardware initialization**

1- scheda di acquisizione  
.AD2() -> connect to hardware Analog Discovery 2 (scheda di acquisizione)
ad2.hdwf  -> identification number of my scheda di acquisizione  
iconfig lets me choose how to allocate resources
=1 (default): buffer is 8192
=3 : buffer is double that, but digital works bad???  
.Scope(ad2.hdwf) -> initialize the oscilloscope (connect it to the scheda)

2- oscilloscope  
frequenza di sampling: can only be a fraction of 100M (ok 50, not ok 60)  
how many points to acquire: keep it smaller than the buffer of my config  

**Channels**

1- range  
can only accept: 
5 -> [-2.5, +2.5]    
50 -> [-25, +25]  
if both are enough, pick 5, it's more precise  
to change range, add an offset  

2- average  
the average means that:  
if i picked sampling rate 50M (max is 100M)  
it still samples at 100M, but only outputs one number every 2, which is the average of the 2  
so it's importante if picked a low sampling rate (else, it just ignores the numbers that come too fast, don't use them to make the average)  

3- offset   
it translates my range (but the width of the range can still be the same as before)   
eg. range=5, offset=2.5 -> [0,5]  

**Acquisition**

scope.sample() stores the results in the numpy arrayw:    
scope.time.vals  
scope.ch1.vals  
scope.ch2.vals  

**Trigger**

1- isteresi  
serve a evitare che il trigger scatti più volte quando il segnale oscilla o è rumoroso vicino alla soglia  
hist = 0.1 significa che dopo uno scatto, il segnale deve allontanarsi dallo soglia di 0.1V prima che possa scattare di nuovo  

2- canale si cui metterlo  
sour = tdwf.trigsrcCh2

3- soglia  
level = 1, cioè 1V  

**Function generator**

i have 2 exit pins: w1 and w2  
for each property, must specify what channel it is for  

put it to sleep after setting the waveform, bc needs time to reach it  
don't sample right away  

**Set wave properties**

command | what | default
---|---|---
offs | offset | 0 V <br> starts from 0, not from the bottom of the range set to the oscilloscope
ampl | amplitude (tra picchi? o la metà?) | 1 V  
freq | frequency | 1 kHz
phi | phase | 0 deg
duty | duty cycle (summetry) | 50 %
func | type of waveform <br> the possible values below | sin

**Set waveforms**

command | what
---|---
funcDC | DC, horizontal line
funcSine | sinusoidal (default)
funcSquare
funcTriangle
funcCustom | pass to the channel w1.data(), containing an array of points between -1 and +1, describing the wave

### Pins

**Pins**

arancione pieno: CH1 (vcc)  
arancione con linea bianca: CH1 (GND)  

giallo pieno: generatore d'onda, w1  
giallo con linea bianca: generatore d'onda, w2  

nero: GND  

**AD2 non raggiungibile**

if i stop an acquisition with Ctrl-C, then i have to run by hand the disconnectio:  
ad2.close()  

**If jupyter notebook not running**

click 'Restart' on the top bar, on the righ of 'Run All'  
it restarts the krnel, then try running it again  